<a href="https://colab.research.google.com/github/nisal-eng/Statistical-Learning-e22206/blob/main/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Bayesian Estimation of a User Ability Parameter from Item Responses (2PL IRT)1. Visualizing the Mechanics of the 2PL IRT ModelThe two-parameter logistic (2PL) model expresses the probability of a correct response as:$$P(X_{ij} = 1 \mid \theta) = \frac{1}{1 + e^{-a_j(\theta - b_j)}}$$The difficulty parameter $b_j$ acts as a horizontal shift factor. Mathematically, when $\theta = b_j$, the exponent becomes $0$, yielding $P(X_{ij} = 1 \mid \theta) = 0.5$.Increasing $b_j$ shifts the Item Characteristic Curve (ICC) to the right, meaning a higher latent ability $\theta$ is required to achieve the same $50\%$ probability of success.Decreasing $b_j$ shifts the curve to the left, making it easier for lower-ability individuals to solve the item.The code below implements this visualization dynamically using Plotly.

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the grid for latent ability
theta = np.linspace(-4, 4, 200)

def irt_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Parameters for visualization
# High discrimination (a = 2.0) paired with three different difficulties
curves = [
    {"a": 2.0, "b": -1.5, "name": "a=2.0, b=-1.5 (Easy)"},
    {"a": 2.0, "b": 0.0, "name": "a=2.0, b=0.0 (Medium)"},
    {"a": 2.0, "b": 1.5, "name": "a=2.0, b=1.5 (Hard)"},
    # Low discrimination (a = 0.6) paired with medium difficulty
    {"a": 0.6, "b": 0.0, "name": "a=0.6, b=0.0 (Low Discrim)"}
]

fig = go.Figure()
for c in curves:
    fig.add_trace(go.Scatter(x=theta, y=irt_2pl(theta, c["a"], c["b"]), name=c["name"], mode='lines'))

fig.update_layout(
    title="Item Characteristic Curves under the 2PL IRT Model",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(X=1|θ)",
    template="plotly_white"
)
fig.show()

2. Sequential Likelihood and Joint HistoryFor a single standalone trial at step $t$, the response $X_{i,t} \in \{0, 1\}$ follows a Bernoulli distribution conditioned on $P(X_{i,t} = 1 \mid \theta)$. The single-step likelihood contribution is:$$L(X_{i,t} \mid \theta) = \left[P(X_{i,t}=1 \mid \theta)\right]^{X_{i,t}} \left[1 - P(X_{i,t}=1 \mid \theta)\right]^{1 - X_{i,t}}$$Assuming responses are conditionally independent given the user's true ability $\theta$, the joint likelihood function for the running history vector $\mathbf{X}_{i,t} = [X_{i,1}, X_{i,2}, \dots, X_{i,t}]^T$ decomposes into the product of individual steps:$$L(\mathbf{X}_{i,t} \mid \theta) = \prod_{\tau=1}^t \left[P(X_{i,\tau}=1 \mid \theta)\right]^{X_{i,\tau}} \left[1 - P(X_{i,\tau}=1 \mid \theta)\right]^{1 - X_{i,\tau}}$$3. Mathematical Formulation of the Running UpdateUsing Bayes' theorem sequentially, the posterior density at step $t$ is proportional to the product of the likelihood at step $t$ and the posterior distribution from step $t-1$:$$p(\theta \mid \mathbf{X}_{i,t}) \propto p(\theta \mid \mathbf{X}_{i,t-1}) \cdot L(X_{i,t} \mid \theta)$$Expanding the formulation explicitly:$$p(\theta \mid \mathbf{X}_{i,t}) = \frac{p(\theta \mid \mathbf{X}_{i,t-1}) \cdot \left[\frac{1}{1 + e^{-a_t(\theta - b_t)}}\right]^{X_{i,t}} \left[\frac{e^{-a_t(\theta - b_t)}}{1 + e^{-a_t(\theta - b_t)}}\right]^{1 - X_{i,t}}}{\int_{-\infty}^{\infty} p(\theta \mid \mathbf{X}_{i,t-1}) \cdot L(X_{i,t} \mid \theta) \, d\theta}$$4. Dynamic Shifting MechanicsWhen a user correctly answers ($X_{i,t} = 1$) a highly difficult item ($b_t \gg 0$), the prior distribution $p(\theta \mid \mathbf{X}_{i,t-1})$ is multiplied by an increasing logistic function that evaluates close to $0$ for all $\theta < b_t$, rising sharply as $\theta$ approaches and exceeds $b_t$.Mathematically, this zero-weights the probability density across the lower ability ranges. Because the likelihood function provides strong evidence that the user's ability exceeds $b_t$, the product shifts the peak (mode) of the posterior density distribution dramatically to the right relative to the previous step.5. Tracking Certainty and Sharpness via DiscriminationThe item discrimination parameter $a_t$ represents the maximum slope of the ICC (occurring at $\theta = b_t$).High Discrimination (Large $a_t$): The logistic curve transitions steeply from $0$ to $1$. The derivative of the log-likelihood with respect to $\theta$ is large near $b_t$, injecting substantial Fisher information. This acts as a highly selective numerical filter, radically narrowing the variance and increasing the sharpness of the updated posterior distribution.Low Discrimination (Small $a_t$): The logistic curve is flat, and the item's response probability depends weakly on $\theta$. Consequently, the likelihood function is uniform over a broad range, adding minimal information and causing little to no change in the variance or sharpness of the distribution.6. Numerical Implementation of a Running GridBecause the logistic function is non-linear in $\theta$, the posterior does not belong to a standard conjugate family and must be approximated numerically.Grid Initialization: Define a discrete vector of evaluation points $\mathbf{\Theta} = [\theta_1, \theta_2, \dots, \theta_M]$ over a bounded interval (e.g., $[-4, 4]$) with uniform spacing $\Delta\theta$. Initialize the prior array $\mathbf{p}_0$ by evaluating the Gaussian PDF $\mathcal{N}(0, 1)$ at each node.Sequential Likelihood Scaling: Upon observing $X_{i,t}$ for an item with known parameters $(a_t, b_t)$, compute an unnormalized vector $\mathbf{\tilde{p}}_t$ via an element-wise product:$$\tilde{p}_t(\theta_m) = p_{t-1}(\theta_m) \cdot \left(\frac{1}{1 + e^{-a_t(\theta_m - b_t)}}\right)^{X_{i,t}} \left(1 - \frac{1}{1 + e^{-a_t(\theta_m - b_t)}}\right)^{1 - X_{i,t}}$$Numerical Normalization: Normalize the vector using Riemann summation or the trapezoidal rule to guarantee that the total area under the curve equals 1:$$p_t(\theta_m) = \frac{\tilde{p}_t(\theta_m)}{\sum_{k=1}^M \tilde{p}_t(\theta_k) \Delta\theta}$$7. Numerical Simulation and Performance TrackingThe following Python script simulates a user with a true latent ability $\theta_0 = 1.5$ encountering $N = 50$ random items, evaluating both the running Posterior Mean and the Maximum A Posteriori (MAP) estimates.

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Seed for reproducibility
np.random.seed(42)

# Simulation Parameters
true_theta = 1.5
num_items = 50
grid_points = 500
theta_grid = np.linspace(-4, 4, grid_points)
delta_theta = theta_grid[1] - theta_grid[0]

# Initialize Prior: Standard Normal
posterior_grid = np.exp(-0.5 * theta_grid**2) / np.sqrt(2 * np.pi)

# Storage for tracking metrics
mean_track = []
map_track = []

# Generate random items
a_params = np.random.uniform(0.5, 2.0, num_items)
b_params = np.random.uniform(-2.0, 2.0, num_items)

# Sequential updates
for t in range(num_items):
    a = a_params[t]
    b = b_params[t]

    # Calculate true success probability and simulate response
    p_success = 1 / (1 + np.exp(-a * (true_theta - b)))
    x_t = 1 if np.random.uniform(0, 1) < p_success else 0

    # Calculate likelihood across the entire grid
    p_grid = 1 / (1 + np.exp(-a * (theta_grid - b)))
    likelihood = (p_grid ** x_t) * ((1 - p_grid) ** (1 - x_t))

    # Update and normalize
    posterior_grid = posterior_grid * likelihood
    total_area = np.sum(posterior_grid) * delta_theta
    posterior_grid /= total_area

    # Calculate point estimates
    post_mean = np.sum(theta_grid * posterior_grid) * delta_theta
    post_map = theta_grid[np.argmax(posterior_grid)]

    mean_track.append(post_mean)
    map_track.append(post_map)

# Visualization
steps = list(range(1, num_items + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=mean_track, mode='lines+markers', name='Expected Value (Mean)'))
fig.add_trace(go.Scatter(x=steps, y=map_track, mode='lines+markers', name='MAP Estimate'))
fig.add_trace(go.Scatter(x=[1, num_items], y=[true_theta, true_theta],
                         mode='lines', name='True Ability (θ₀=1.5)', line=dict(dash='dash', color='black')))

fig.update_layout(
    title="2PL IRT Latent Ability Estimation Timeline",
    xaxis_title="Item Step (t)",
    yaxis_title="Estimated Ability Parameter",
    template="plotly_white"
)
fig.show()

Analysis & InterpretationAs the item step counter $t$ increases, the distance between both estimators and the true parameter value $\theta_0$ decreases, showing clear asymptotic convergence. Early updates exhibit high volatility, as individual correct or incorrect answers cause large shifts in the distribution.As more responses are integrated, the cumulative evidence narrows the variance of the posterior distribution. This reduction in variance implies a steady increase in the platform's measurement confidence, making the estimators increasingly resilient to individual anomalous responses (e.g., a lucky guess or a careless mistake).

Part 2: Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates
1. Structural Probability and Properties of the Beta Distribution
The Beta distribution serves as a flexible prior for parameters bounded between 0 and 1. The code block below visualizes its probability density function under different parameters.

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_domain = np.linspace(0, 1, 300)

states = [
    {"alpha": 1.0, "beta": 1.0, "name": "Uninformative: α=1, β=1"},
    {"alpha": 2.0, "beta": 5.0, "name": "Right-Skewed (Low CTR): α=2, β=5"},
    {"alpha": 5.0, "beta": 2.0, "name": "Left-Skewed (High CTR): α=5, β=2"}
]

fig = go.Figure()
for s in states:
    pdf_vals = stats.beta.pdf(theta_domain, s["alpha"], s["beta"])
    fig.add_trace(go.Scatter(x=theta_domain, y=pdf_vals, name=s["name"], mode='lines'))

fig.update_layout(
    title="Beta Distribution Profiles",
    xaxis_title="Click-Through Rate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig.show()

Interpretation of Center of MassWhen $\alpha = \beta$, the distribution is perfectly symmetrical. For $\alpha=\beta=1$, the density is uniform, spreading mass equally across the entire domain.When $\beta > \alpha$, the center of mass shifts toward 0 (right-skewed). This indicates a prior belief that the advertisement is highly likely to have a low click-through rate.When $\alpha > \beta$, the density shifts toward 1 (left-skewed), concentrating mass in the high-performance region.2. Sequential Likelihood and Joint HistoryThe likelihood contribution of a single Bernoulli outcome $X_t \in \{0, 1\}$ given the true conversion parameter $\theta$ is:$$L(X_t \mid \theta) = \theta^{X_t}(1-\theta)^{1-X_t}$$For a historical stream of conditionally independent observations $\mathbf{X}_t = [X_1, X_2, \dots, X_t]^T$, the joint likelihood function equals the product of the individual likelihoods:$$L(\mathbf{X}_t \mid \theta) = \prod_{\tau=1}^t \theta^{X_\tau}(1-\theta)^{1-X_\tau} = \theta^{\sum_{\tau=1}^t X_\tau}(1-\theta)^{t - \sum_{\tau=1}^t X_\tau}$$3. Closed-Form Analytical Updates (Beta-Binomial Conjugacy)We wish to compute the exact recursive formulation for the posterior density at step $t$:$$p(\theta \mid \mathbf{X}_t) \propto p(\theta \mid \mathbf{X}_{t-1}) \cdot L(X_t \mid \theta)$$Given that the prior at step $t$ is the posterior from the previous milestone, it follows a Beta distribution:$$p(\theta \mid \mathbf{X}_{t-1}) = \frac{1}{\text{B}(\alpha_{t-1}, \beta_{t-1})} \theta^{\alpha_{t-1}-1}(1-\theta)^{\beta_{t-1}-1}$$Substituting this prior and the single-step Bernoulli likelihood into Bayes' theorem:$$p(\theta \mid \mathbf{X}_t) \propto \left[ \theta^{\alpha_{t-1}-1}(1-\theta)^{\beta_{t-1}-1} \right] \cdot \left[ \theta^{X_t}(1-\theta)^{1-X_t} \right]$$Combining exponents:$$p(\theta \mid \mathbf{X}_t) \propto \theta^{(\alpha_{t-1} + X_t) - 1}(1-\theta)^{(\beta_{t-1} + 1 - X_t) - 1}$$This functional form matches the kernel of a Beta distribution. Therefore, the posterior distribution is certified to remain within the Beta family, proving Beta-Binomial conjugacy. The exact updating equations for the shape parameters are:$$\alpha_t = \alpha_{t-1} + X_t$$$$\beta_t = \beta_{t-1} + (1 - X_t)$$The analytical expected value (Posterior Mean) at step $t$ is:$$\mathbb{E}[\theta \mid \mathbf{X}_t] = \frac{\alpha_t}{\alpha_t + \beta_t} = \frac{\alpha_{t-1} + X_t}{\alpha_{t-1} + \beta_{t-1} + 1}$$4. Dynamic Shifting Mechanics: Conjugate vs. Non-ConjugateAnalytical Update: If $X_t = 1$ (click), $\alpha$ increments by 1 while $\beta$ remains unchanged, shifting the mode to the right. If $X_t = 0$ (no-click), $\beta$ increments by 1, shifting the mode to the left.Conjugate vs. Non-Conjugate Contrast: In this Beta-Binomial framework, updates require only basic scalar additions ($\alpha + 1$ or $\beta + 1$). In contrast, non-conjugate structures like the 2PL IRT model lack an analytical solution. They force the system to perform computationally demanding numerical integrations or grid evaluations at every step to update the posterior distribution.5. Running Point EstimatorsFrom the updated parameters $\alpha_t$ and $\beta_t$, the exact point estimators are computed directly using closed-form algebraic solutions:Running Posterior Mean:$$\text{Mean}_t = \frac{\alpha_t}{\alpha_t + \beta_t}$$Running Maximum A Posteriori (MAP):$$\text{MAP}_t = \frac{\alpha_t - 1}{\alpha_t + \beta_t - 2} \quad (\text{valid for } \alpha_t, \beta_t > 1)$$6. Performance Tracking and Convergence Analysis

In [ ]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(101)

# Simulation setup
true_ctr = 0.32
total_impressions = 200

# Base Prior parameters (Uniform state)
alpha_t = 1.0
beta_t = 1.0

mean_history = []
map_history = []

for t in range(1, total_impressions + 1):
    # Simulate single Bernoulli observation
    x_t = 1 if np.random.uniform(0, 1) < true_ctr else 0

    # Direct closed-form conjugate update
    alpha_t += x_t
    beta_t += (1 - x_t)

    # Store point estimates
    mean_history.append(alpha_t / (alpha_t + beta_t))

    if alpha_t > 1 and beta_t > 1:
        map_history.append((alpha_t - 1) / (alpha_t + beta_t - 2))
    else:
        map_history.append(0.5) # Default fallback if undefined

# Visualization
idx = list(range(1, total_impressions + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=idx, y=mean_history, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=idx, y=map_history, mode='lines', name='MAP Estimate'))
fig.add_trace(go.Scatter(x=[1, total_impressions], y=[true_ctr, true_ctr],
                         mode='lines', name='True CTR (0.32)', line=dict(dash='dash', color='red')))

fig.update_layout(
    title="Sequential Beta-Binomial CTR Estimation Tracking",
    xaxis_title="Impressions Observed (t)",
    yaxis_title="Estimated Click-Through Rate",
    template="plotly_white"
)
fig.show()

Analysis & InterpretationAs the sample size $t$ approaches $200$, the distance between the sequential point estimates and the true conversion parameter $\theta_0 = 0.32$ decreases significantly.Because the initial prior parameters were small ($\alpha_0 = \beta_0 = 1$), the prior carries an informational weight equivalent to only two observations. Consequently, the data quickly dominates the prior structure. As evidence accumulates, the posterior distribution narrows around the true parameter value, showing that the system rapidly corrects for initial uncertainty.Part 3: Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates1. Prior Belief BoundariesThe structural stiffness efficiency parameter $\theta$ is physically bounded to the domain $[0, 1]$. The initial prior is modeled using a bounded Beta distribution: $p_0(\theta) \sim \text{Beta}(12, 1.5)$.

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_shm = np.linspace(0.001, 0.999, 400)
prior_shm = stats.beta.pdf(theta_shm, 12, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_shm, y=prior_shm, mode='lines', name='Prior Beta(12, 1.5)'))
fig.update_layout(
    title="Initial Structural Health Prior Distribution",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Density",
    template="plotly_white"
)
fig.show()

The analytical expected prior stiffness efficiency is:$$\mathbb{E}[\theta] = \frac{\alpha}{\alpha + \beta} = \frac{12}{12 + 1.5} = \frac{12}{13.5} \approx 0.8889$$Engineering JustificationThis distribution concentrates its probability mass near $1.0$, making it an appropriate choice for structural health monitoring. It reflects the prior engineering expectation that a newly deployed or recently inspected aerospace component is highly likely to be structurally pristine, while still leaving a small probability for early degradation.2. Structural Likelihood FormulationThe physics degradation framework yields the following observation equation:$$Y_t = Y_0 \theta^{\gamma} e^{\epsilon_t}, \quad \epsilon_t \sim \mathcal{N}(0, \sigma^2)$$Taking the natural logarithm transforms this into a linear equation with additive Gaussian noise:$$\log(Y_t) = \log(Y_0) + \gamma \log(\theta) + \epsilon_t \implies \log(Y_t) \sim \mathcal{N}\left(\log(Y_0) + \gamma \log(\theta), \, \sigma^2\right)$$Using the transformation of variables rule, the probability density function of the log-normal measurement $Y_t$ given $\theta$ is:$$p(Y_t \mid \theta) = p_{\log(Y_t)}(\log(Y_t)) \cdot \left\vert{} \frac{d \log(Y_t)}{d Y_t} \right\vert{} = \frac{1}{Y_t \sigma \sqrt{2\pi}} \exp \left( -\frac{\left(\log(Y_t) - \log(Y_0) - \gamma \log(\theta)\right)^2}{2\sigma^2} \right)$$Assuming independent measurement errors over the timeline, the joint likelihood function for the running history vector $\mathbf{Y}_t = [Y_1, Y_2, \dots, Y_t]^T$ is:$$L(\mathbf{Y}_t \mid \theta) = \prod_{\tau=1}^t \frac{1}{Y_\tau \sigma \sqrt{2\pi}} \exp \left( -\frac{\left(\log(Y_\tau) - \log(Y_0) - \gamma \log(\theta)\right)^2}{2\sigma^2} \right)$$3. Non-Conjugate Grid Update FormulationAn exact analytical closed-form solution for the posterior density does not exist. The structural parameter $\theta$ appears inside a logarithm ($\gamma \log(\theta)$) within the exponent of the log-normal likelihood. When multiplied by the polynomial kernel of the Beta prior ($\theta^{\alpha-1}(1-\theta)^{\beta-1}$), the exponents cannot be combined into a standard distribution family.Thus, we must compute the posterior numerically using the following recursive formulation:$$p(\theta \mid \mathbf{Y}_t) \propto p(\theta \mid \mathbf{Y}_{t-1}) \cdot \frac{1}{Y_t} \exp \left( -\frac{\left(\log(Y_t) - \log(Y_0) - \gamma \log(\theta)\right)^2}{2\sigma^2} \right)$$4. Running Point Estimates via Definite IntegralsBecause the normalized posterior density function $p(\theta \mid \mathbf{Y}_t)$ is non-conjugate, we compute its point estimates over the bounded physical domain using definite numerical integrations:Running Posterior Mean:$$\text{Mean}_t = \int_{0}^{1} \theta \cdot p(\theta \mid \mathbf{Y}_t) \, d\theta$$Running Maximum A Posteriori (MAP):$$\text{MAP}_t = \arg\max_{\theta \in [0, 1]} p(\theta \mid \mathbf{Y}_t)$$5. Algorithmic Grid Approximation and NormalizationTo track the system state without an analytical solution, we maintain the distribution over a discrete grid:Grid Setup: Discretize the domain $[0, 1]$ into a fine linear grid $\mathbf{\Theta} = [\theta_1, \theta_2, \dots, \theta_M]$. To avoid evaluation errors with $\log(0)$, set the lower boundary slightly above zero (e.g., $\theta_1 = 10^{-4}$).Likelihood Evaluation: When a new measurement $Y_t$ arrives, compute the unnormalized likelihood array across the grid:$$\mathbf{L}_t = \exp \left( -\frac{\left(\log(Y_t) - \log(Y_0) - \gamma \log(\mathbf{\Theta})\right)^2}{2\sigma^2} \right)$$Prior Multiplication: Update the unnormalized posterior array: $\mathbf{\tilde{p}}_t = \mathbf{p}_{t-1} \odot \mathbf{L}_t$, where $\odot$ denotes element-wise multiplication.Trapezoidal Integration: Compute the normalizing constant (the total area under the unnormalized curve) using the trapezoidal rule:$$I = \text{np.trapezoid}(\mathbf{\tilde{p}}_t, \mathbf{\Theta})$$Final Normalization: Divide the unnormalized array by the integral value to obtain the valid density: $\mathbf{p}_t = \frac{\mathbf{\tilde{p}}_t}{I}$.6. Performance Tracking and Degradation Convergence AnalysisThe code below simulates an aircraft component that has sustained structural damage, dropping its true remaining stiffness efficiency to $\theta_0 = 0.68$.

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Seed for consistency
np.random.seed(42)

# Physical constants
Y_0 = 100.0
gamma = 1.2
sigma = 0.15
true_theta = 0.68
num_steps = 30

# Grid infrastructure
M = 1000
theta_grid = np.linspace(0.001, 0.999, M)

# Initialize prior: Beta(12, 1.5)
post_density = stats.beta.pdf(theta_grid, 12, 1.5)
# Initial normalization override
post_density /= np.trapezoid(post_density, theta_grid)

# Storage matrices
mean_history = []
map_history = []
milestones = {1: None, 5: None, 15: None, 30: None}

# Generate noisy sensor stream based on underlying log-normal physics
noise = np.random.normal(0, sigma, num_steps)
sensor_stream = Y_0 * (true_theta ** gamma) * np.exp(noise)

# Sequential updates
for t in range(1, num_steps + 1):
    Y_t = sensor_stream[t-1]

    # Evaluate likelihood contribution across the grid
    log_diff = np.log(Y_t) - np.log(Y_0) - gamma * np.log(theta_grid)
    likelihood = np.exp(-0.5 * (log_diff ** 2) / (sigma ** 2))

    # Update and normalize using the trapezoidal rule
    post_density = post_density * likelihood
    area = np.trapezoid(post_density, theta_grid)
    post_density /= area

    # Store milestone densities for plotting
    if t in milestones:
        milestones[t] = post_density.copy()

    # Calculate point estimates
    current_mean = np.trapezoid(theta_grid * post_density, theta_grid)
    current_map = theta_grid[np.argmax(post_density)]

    mean_history.append(current_mean)
    map_history.append(current_map)

# Plot 1: Full Posterior Density Evolution Profiles
fig_curves = go.Figure()
fig_curves.add_trace(go.Scatter(x=theta_grid, y=stats.beta.pdf(theta_grid, 12, 1.5), name='Initial Prior', mode='lines'))
for k, v in milestones.items():
    fig_curves.add_trace(go.Scatter(x=theta_grid, y=v, name=f'Step {k} Posterior', mode='lines'))
fig_curves.update_layout(
    title="Posterior Density Evolution over Time",
    xaxis_title="Stiffness Efficiency (θ)",
    yaxis_title="Density",
    template="plotly_white"
)
fig_curves.show()

# Plot 2: Convergence History Timeline
steps = list(range(1, num_steps + 1))
fig_track = go.Figure()
fig_track.add_trace(go.Scatter(x=steps, y=mean_history, name='Posterior Mean', mode='lines+markers'))
fig_track.add_trace(go.Scatter(x=steps, y=map_history, name='MAP Estimate', mode='lines+markers'))
fig_track.add_trace(go.Scatter(x=[1, num_steps], y=[true_theta, true_theta],
                               name='True Efficiency (0.68)', line=dict(dash='dash', color='black'), mode='lines'))
fig_track.update_layout(
    title="Convergence Timeline of Bounded Grid Estimators",
    xaxis_title="Inspection Step (t)",
    yaxis_title="Estimated Efficiency",
    template="plotly_white"
)
fig_track.show()

Analysis & InterpretationOvercoming the Prior: The tracking system requires roughly $5$ to $8$ continuous sensor readings to completely overcome the optimistic "healthy" prior and pull the posterior distribution down to isolate the true $68\%$ damage state.Narrowing of Density Curves: As time progresses, the density profiles become significantly narrower and sharper. This narrowing indicates a reduction in uncertainty regarding the estimated structural degradation. In real-world engineering contexts, a sharp, narrow distribution allows operators to set tight, reliable safety thresholds, preventing catastrophic failures without triggering premature, costly maintenance alerts.Part 4: Gaussian Mixture Clustering as Conditional Updating1. Deriving the Marginal DensityWe are given the conditional model for data point $x_i$ belonging to cluster $k$:$$p(x_i \mid z_{ik}=1) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$The prior probability of selecting cluster $k$ is given by $p(z_{ik}=1) = \phi_k$. By applying the law of total probability across all $K$ mutually exclusive hidden categories, the marginal density $p(x_i \mid \Psi)$ is derived as follows:$$p(x_i \mid \Psi) = \sum_{k=1}^K p(z_{ik}=1) \cdot p(x_i \mid z_{ik}=1) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$InterpretationThis formulation is called a Gaussian mixture density because the total marginal probability profile is formed by a convex combination (a weighted sum) of $K$ distinct multivariate normal distributions, where the weights $\phi_k$ are non-negative and sum to 1.2. Deriving the Posterior Cluster ProbabilityFor an observed data coordinate vector $x_i$, the conditional probability that it belongs to a specific cluster $k$ is given by Bayes' rule:$$p(z_{ik}=1 \mid x_i, \Psi) = \frac{p(z_{ik}=1) \cdot p(x_i \mid z_{ik}=1, \Psi)}{p(x_i \mid \Psi)}$$Substituting the prior assignment parameter $\phi_k$, the cluster's Gaussian profile, and the derived marginal density:$$p(z_{ik}=1 \mid x_i, \Psi) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)} \equiv \gamma_{ik}$$InterpretationThe responsibility $\gamma_{ik}$ represents the posterior probability of cluster membership. It quantifies the platform's updated belief that data point $x_i$ was generated by cluster $k$, combining the prior probability $\phi_k$ with the empirical likelihood from the data.3. One-Hot Encoding of the Latent Cluster VariableLet $\mathbf{z}_i = [z_{i1}, z_{i2}, \dots, z_{iK}]^T$ be a one-hot encoded vector where exactly one element equals 1 and all others equal 0. Since $z_{ik}$ is a binary indicator variable, its expected value is simply its probability of being 1:$$\mathbb{E}[z_{ik} \mid x_i, \Psi] = 1 \cdot p(z_{ik}=1 \mid x_i, \Psi) + 0 \cdot p(z_{ik}=0 \mid x_i, \Psi) = \gamma_{ik}$$Stacking these expectations into a vector format yields:$$\mathbb{E}[\mathbf{z}_i \mid x_i, \Psi] = \begin{bmatrix} \mathbb{E}[z_{i1} \mid x_i, \Psi] \\ \mathbb{E}[z_{i2} \mid x_i, \Psi] \\ \vdots \\ \mathbb{E}[z_{iK} \mid x_i, \Psi] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$ConclusionThis proves that the soft cluster assignment vector in a Gaussian Mixture Model (GMM) is exactly equal to the conditional expectation of the latent indicator vector given the observed data:$$\mathbb{E}[\mathbf{z}_i \mid x_i, \Psi] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$$4. From Soft Assignment to Hard ClusteringSoft Clustering: Assigns a data point a continuous probability vector $\mathbb{E}[\mathbf{z}_i \mid x_i, \Psi] = [\gamma_{i1}, \dots, \gamma_{iK}]^T$, reflecting the fractional confidence that the point belongs to each cluster.Hard Clustering: Replaces this probability distribution with a deterministic decision by assigning the point entirely to the single cluster with the highest posterior responsibility: $k^* = \arg\max_{k} \gamma_{ik}$.5. Conditional Expectation of the Observation Given the ClusterBy definition, conditioning on $z_{ik}=1$ isolates cluster $k$. The expected value of a variable following a multivariate normal distribution $\mathcal{N}(\mu_k, \Sigma_k)$ is its mean vector:$$\mathbb{E}[x_i \mid z_{ik}=1] = \mu_k$$Therefore, $\mu_k$ represents the geometric center of cluster $k$.$\mathbb{E}[\mathbf{z}_i \mid x_i, \Psi]$ maps a fixed data point to a probability distribution across the clusters (soft cluster assignment).$\mathbb{E}[x_i \mid z_{ik}=1]$ identifies the expected location in the feature space for a given cluster (cluster center).6. The Complete-Data LikelihoodIf the hidden cluster assignments $z_{ik}$ were directly observable, the complete-data likelihood would be written as:$$L_c(\Psi) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$Taking the natural logarithm transforms the products into sums, yielding the complete-data log-likelihood $\ell_c$:$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$Optimization AdvantageIf the $z_{ik}$ parameters were known, this optimization problem would decouple. The global maximum could be found analytically using standard Maximum Likelihood Estimation (MLE) for independent subsets, bypassing the need for iterative algorithms.7. The EM Interpretation as a Conditional UpdateBecause the indicators $z_{ik}$ are hidden, the Expectation-Maximization (EM) algorithm computes the expected value of the complete-data log-likelihood with respect to the conditional distribution of the latent variables, evaluated using the current parameter estimates $\Psi^{(t)}$.In the E-step, we replace the unobserved variables $z_{ik}$ with their conditional expectations:$$\mathbb{E}[z_{ik} \mid x_i, \Psi^{(t)}] = \gamma_{ik}^{(t)}$$This yields the objective function $Q$:$$Q(\Psi \mid \Psi^{(t)}) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik}^{(t)} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$The E-step can be interpreted as a conditional update of cluster membership probabilities, re-evaluating the responsibilities $\gamma_{ik}$ for every data point based on the current model parameters.8. Parameter Updates (M-Step)In the M-step, we maximize the objective function $Q$ to update the parameters.Update for Mixing Weights ($\phi_k$)We maximize $Q$ subject to the constraint $\sum_{k=1}^K \phi_k = 1$ using a Lagrange multiplier $\lambda$:$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \lambda \left( 1 - \sum_{k=1}^K \phi_k \right)$$Taking the partial derivative with respect to $\phi_k$ and setting it to 0:$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda}$$Summing both sides over $k$ reveals that $\lambda = n$. This gives the standard update equation:$$\phi_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik}}{n}$$Update for Cluster Means ($\mu_k$)Taking the derivative of $Q$ with respect to $\mu_k$:$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1} (x_i - \mu_k) = 0 \implies \sum_{i=1}^n \gamma_{ik} x_i = \left( \sum_{i=1}^n \gamma_{ik} \right) \mu_k$$Solving for $\mu_k$:$$\mu_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik} x_i}{\sum_{i=1}^n \gamma_{ik}}$$Update for Covariances ($\Sigma_k$)Taking the derivative with respect to $\Sigma_k^{-1}$ and solving yields:$$\Sigma_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T}{\sum_{i=1}^n \gamma_{ik}}$$Fractional Membership Weight RoleThe responsibility $\gamma_{ik}$ acts as a continuous weight. Data points with high responsibility for cluster $k$ contribute strongly to its updated mean and covariance, while points with low responsibility are effectively ignored.9. Methodological Synthesis SummaryGaussian Mixture Model clustering is fundamentally an iterative process of conditional updates:The mixing weight $\phi_k$ defines the prior probability of cluster $k$.The Gaussian PDF $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ measures how well the data point $x_i$ fits cluster $k$.The responsibility $\gamma_{ik}$ combines these to form the posterior probability of cluster membership.The M-step updates the cluster parameters ($\phi_k, \mu_k, \Sigma_k$) using these posterior probabilities as weights.This loop guarantees that GMM clustering functions as an expectation-driven probabilistic framework rather than a heuristic geometric partitioning tool.10. Computational Simulation and Out-of-Sample ValidationThe code block below implements the GMMFinancialSegmenter class. It processes real-world financial data, fits a Gaussian Mixture Model using the EM algorithm, evaluates out-of-sample performance, and visualizes the results.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import plotly.express as px

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(n_components=n_components, random_state=42, init_params='kmeans')

    def prepare_data(self, df, feature_cols):
        # Drop missing values and extract features
        data = df[feature_cols].dropna()
        X = data.values

        # Train-test split (80/20)
        X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

        # Standardize features
        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)
        return self.X_train_scaled, self.X_test_scaled

    def fit(self):
        self.gmm.fit(self.X_train_scaled)
        print(f"Convergence State: {self.gmm.converged_}")
        print(f"Iterations Required: {self.gmm.n_iter_}")

    def evaluate(self):
        test_ll = self.gmm.score(self.X_test_scaled)
        print(f"Average Log-Likelihood on Test Set: {test_ll:.4f}")
        return test_ll

    def plot_density_heatmap(self):
        fig = px.density_heatmap(
            x=self.X_train_scaled[:, 0], y=self.X_train_scaled[:, 1],
            marginal_x="histogram", marginal_y="histogram",
            labels={'x': 'Scaled Purchases', 'y': 'Scaled Credit Limit'},
            title="Empirical 2D Density Heatmap (Training Set)"
        )
        fig.show()

    def _generate_contour_grid(self):
        x_min, x_max = self.X_train_scaled[:, 0].min() - 0.5, self.X_train_scaled[:, 0].max() + 0.5
        y_min, y_max = self.X_train_scaled[:, 1].min() - 0.5, self.X_train_scaled[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Calculate responsibilities across the grid
        resp = self.gmm.predict_proba(grid_points)
        max_resp = np.max(resp, axis=1).reshape(xx.shape)
        return xx, yy, max_resp

    def plot_training_assignments(self):
        xx, yy, max_resp = self._generate_contour_grid()
        train_labels = self.gmm.predict(self.X_train_scaled)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=xx[0, :], y=yy[:, 0], z=max_resp, colorscale='Viridis',
                                 contours_coloring='heatmap', name='Max Responsibility'))
        fig.add_trace(go.Scatter(x=self.X_train_scaled[:, 0], y=self.X_train_scaled[:, 1],
                                 mode='markers', marker=dict(color=train_labels, size=4, line=dict(width=0.5, color='white')),
                                 name='Train Points'))
        fig.update_layout(title="Training Boundary Map & Assignments", xaxis_title="Purchases", yaxis_title="Credit Limit")
        fig.show()

    def plot_test_assignments(self):
        xx, yy, max_resp = self._generate_contour_grid()
        test_labels = self.gmm.predict(self.X_test_scaled)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=xx[0, :], y=yy[:, 0], z=max_resp, colorscale='Viridis',
                                 contours_coloring='heatmap', name='Max Responsibility'))
        fig.add_trace(go.Scatter(x=self.X_test_scaled[:, 0], y=self.X_test_scaled[:, 1],
                                 mode='markers', marker=dict(color=test_labels, size=5, line=dict(width=0.5, color='black')),
                                 name='Test Points'))
        fig.update_layout(title="Out-of-Sample Test Evaluation Map", xaxis_title="Purchases", yaxis_title="Credit Limit")
        fig.show()

# -------------------------------------------------------------
# Execution using synthetic data mimicking the CC GENERAL structure
# -------------------------------------------------------------
# Generate synthetic dataset mirroring PURCHASES and CREDIT_LIMIT distributions
n_samples = 1000
cluster_1 = np.random.multivariate_normal([ -0.8,  -0.5], [[0.15, 0.05], [0.05, 0.20]], int(n_samples * 0.5))
cluster_2 = np.random.multivariate_normal([  1.2,   0.8], [[0.25, 0.10], [0.10, 0.30]], int(n_samples * 0.3))
cluster_3 = np.random.multivariate_normal([ -0.2,   1.5], [[0.20, 0.00], [0.00, 0.15]], int(n_samples * 0.2))

synthetic_data = np.vstack([cluster_1, cluster_2, cluster_3])
mock_df = pd.DataFrame(synthetic_data, columns=['PURCHASES', 'CREDIT_LIMIT'])

# Run GMM pipeline
segmenter = GMMFinancialSegmenter(n_components=3)
segmenter.prepare_data(mock_df, ['PURCHASES', 'CREDIT_LIMIT'])
segmenter.fit()
segmenter.evaluate()

# Display interactive Plotly figures
segmenter.plot_density_heatmap()
segmenter.plot_training_assignments()
segmenter.plot_test_assignments()

Visual Evaluation & Theoretical Bridge
The continuous background contour map displays the maximum posterior responsibility max
k
​
 γ
ik
​
  across the feature space.

In the core regions of each cluster, the background color is solid and bright, showing a responsibility value close to 1.0. Here, the model is highly confident in its assignment.

In the transition zones between clusters, the background color fades. This visual drop in intensity represents regions of cluster ambiguity, directly reflecting the soft assignment expectation vector E[z
i
​
 ∣x
i
​
 ,Ψ]. Rather than drawing rigid boundaries, the GMM naturally quantifies uncertainty near the intersections of different groups.